# Stratification introduction

Adapted from the [summer2 documentation](https://summer2.readthedocs.io)
page `examples/06-stratification-introduction` at commit
`d1537d6188aba85c33c0449b197eef6ad8b03d6c` of
[monash-emu/summer2](https://github.com/monash-emu/summer2).

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

Stratify a compartment map with `Property` + `PropertyMap.stratify`, or stratify
an existing builder with `FlowModel.stratify` (in place; use `copy()` to branch).
Adjust flows with `Multiply` / `Overwrite` via `adjust=` or `adjust_flow`, and
set infectiousness on the `ForceOfInfection` on `FlowModel` (not on a
stratification object).

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import plotly.io as pio

from summer4 import (
    Compartments,
    Dest,
    EntryFlow,
    ExitFlow,
    FlowMass,
    FlowModel,
    Multiply,
    Param,
    Property,
    PropertyData,
    PropertyMap,
    SavePlan,
    SaveRequest,
    Source,
    TransitionFlow,
)
from summer4.epi import ForceOfInfection, MixingMatrix

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

state = Property("state", ("S", "I", "R"))
age = Property("age", ("young", "old"))
severity = Property("severity", ("asymptomatic", "mild", "severe"))
pop = Property("pop", ("all",))

PARAMS = {"beta": 2.0}


def plot_comp(res, title):
    return res["comp"].to_pandas().plot(
        title=title, labels={"index": "time (days)", "value": "people"}
    )


def plot_flow(res, name, title, group=None, side="source"):
    trace = res[name]
    if group is not None:
        trace = trace.sum_over(group, side=side)
    return trace.to_pandas().plot(
        title=title, labels={"index": "time (days)", "value": "people / day"}
    )


def plot_flows(res, names, title, group=None, side="source"):
    frames = []
    for name in names:
        trace = res[name]
        if group is not None:
            trace = trace.sum_over(group, side=side)
        frames.append(trace.to_pandas().add_prefix(f"{name} "))
    return pd.concat(frames, axis=1).plot(
        title=title, labels={"index": "time (days)", "value": "people / day"}
    )


def run(model, y0, t1=20.0, params=None):
    req = {"comp": SaveRequest(Compartments())}
    for flow in model.flows:
        req[flow.name] = SaveRequest(FlowMass(flow=flow.name))
    plan = SavePlan(
        requests=req,
        ts=jnp.linspace(0.0, t1, 201),
    )
    if not isinstance(y0, PropertyData):
        y0 = PropertyData.wrap(model.pmap if hasattr(model, "pmap") else y0.pmap, jnp.asarray(y0))
    return model.compile().run(
        params if params is not None else PARAMS,
        y0,
        t0=0.0,
        t1=t1,
        dt=0.1,
        save=plan,
        solver="euler",
    )


def base_y0(pmap, s=990.0, i=10.0):
    y0 = jnp.zeros(pmap.size)
    n_s = len(pmap.select(state["S"]))
    n_i = len(pmap.select(state["I"]))
    y0 = y0.at[pmap.select(state["S"])].set(s / n_s)
    y0 = y0.at[pmap.select(state["I"])].set(i / n_i)
    return PropertyData.wrap(pmap, y0)


def age_foi(**kwargs):
    """Frequency FOI grouped by age with homogeneous mixing."""
    return ForceOfInfection(
        "infection",
        infectious=state["I"],
        group_by=age,
        mixing=MixingMatrix(age, np.ones((2, 2)), normalize="rows", check_reciprocal=False),
        kind="frequency",
        contact_rate=Param("beta"),
        **kwargs,
    )


# One unstratified SIR; later sections copy() + stratify() + adjust/update.
pmap0 = PropertyMap.from_property(state).stratify(pop)
base = FlowModel(pmap0)
mixing0 = MixingMatrix(pop, np.array([[1.0]]), check_reciprocal=False)
base.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        ForceOfInfection(
            "infection",
            infectious=state["I"],
            group_by=mixing0.prop,
            mixing=mixing0,
            kind="frequency",
            contact_rate=Param("beta"),
        ),
    )
)
base.add_flow(TransitionFlow("recovery", state["I"], state["R"], 1.0 / 3.0))
base.add_flow(ExitFlow("infection_death", state["I"], 0.05))

## No stratification

A regular SIR with infection, recovery, and infection death.

In [ ]:
res0 = run(base, base_y0(pmap0))
assert float(np.max(np.asarray(res0["comp"].select(state["I"]).values.data))) > 10.0
plot_comp(res0, "Unstratified SIR")

## Minimal stratification

Split every compartment into young / old. With homogeneous mixing (FOI still
groups by the dummy `pop` stratum) and equal population shares the aggregate
dynamics match the unstratified model.

In [ ]:
m = base.copy()
m.stratify(age)
pmap_age = m.pmap
y0_age = base_y0(pmap_age)
cm_age = m.compile()
plan_age = SavePlan(
    requests={"comp": SaveRequest(Compartments())},
    ts=jnp.linspace(0.0, 20.0, 201),
)
res = cm_age.run(PARAMS, y0_age, t0=0.0, t1=20.0, dt=0.1, save=plan_age, solver="euler")
assert pmap_age.size == 6
plot_comp(res, "Age-stratified SIR (even split)")

## Importation after stratification

`EntryFlow` into `I` splits absolute imports across age strata.

In [ ]:
m = base.copy()
m.stratify(age)
m.add_flow(EntryFlow("infection_imports", state["I"], 10.0))
res = run(m, base_y0(m.pmap))
i_end = float(np.asarray(res["comp"].select(state["I"]).total().values)[-1])
assert i_end > 0.0
plot_comp(res, "Age strata with importation into I")

## Population distribution

Initial conditions need not be even — here 60% young, 40% old.

In [ ]:
y0 = jnp.zeros(pmap_age.size)
y0 = y0.at[pmap_age.select(state["S"] & age["young"])].set(990.0 * 0.6)
y0 = y0.at[pmap_age.select(state["S"] & age["old"])].set(990.0 * 0.4)
y0 = y0.at[pmap_age.select(state["I"] & age["young"])].set(10.0 * 0.6)
y0 = y0.at[pmap_age.select(state["I"] & age["old"])].set(10.0 * 0.4)
y0 = PropertyData.wrap(pmap_age, y0)
m = base.copy()
m.stratify(age)
res = run(m, y0)
s_y = float(np.asarray(res["comp"].select(state["S"] & age["young"]).values.data)[0, 0])
s_o = float(np.asarray(res["comp"].select(state["S"] & age["old"]).values.data)[0, 0])
assert abs(s_y / s_o - 0.6 / 0.4) < 1e-6
plot_comp(res, "60:40 young:old initial split")

## Flow adjustments

Young people are twice as susceptible; death and recovery rates also differ by
age. Adjustments are flow-owned (`adjust_flow` / `adjust=`), not stratification
methods. The compartment plot shows fewer recovered old people; the flow plot
shows the adjusted infection, recovery, and death rates by age.

In [ ]:
m = base.copy()
m.stratify(age)
m.update_flow("infection", rate=age_foi())
m.adjust_flow("infection", Multiply(2.0, where=age["young"]))
m.adjust_flow("recovery", Multiply(0.5, where=age["young"]))
m.adjust_flow(
    "infection_death",
    Multiply(3.0, where=age["old"]),
    Multiply(0.5, where=age["young"]),
)
res = run(m, y0)
r_old = float(np.asarray(res["comp"].select(state["R"] & age["old"]).values.data)[-1, 0])
r_young = float(np.asarray(res["comp"].select(state["R"] & age["young"]).values.data)[-1, 0])
assert r_old < r_young  # higher death among old → fewer recovered old
plot_comp(res, "Flow adjustments by age")

In [ ]:
death = res["infection_death"].sum_over(age, side="source").to_pandas()
assert "age=young" in death.columns and "age=old" in death.columns
assert float(death["age=old"].max()) > 0.0
plot_flows(
    res,
    ("infection", "recovery", "infection_death"),
    "Adjusted infection, recovery, and death flows by age",
    group=age,
)


## Infectiousness adjustments

Infectiousness is FOI-owned: pass weights into `ForceOfInfection` /
`ForceOfInfection(infectiousness=...)`. Young people are 1.2× as infectious
and twice as susceptible.

In [ ]:
m = base.copy()
m.stratify(age)
m.update_flow(
    "infection",
    rate=age_foi(
        infectiousness={age["young"]: 1.2, age["old"]: 1.0},
        normalize_infectiousness=None,
    ),
)
m.adjust_flow("infection", Multiply(2.0, where=age["young"]))
res = run(m, base_y0(m.pmap))
assert float(np.max(np.asarray(res["comp"].select(state["I"]).values.data))) > 10.0
plot_comp(res, "Infectiousness + susceptibility by age")

In [ ]:
inc = res["infection"].sum_over(age, side="source")
assert "age=young" in inc.to_pandas().columns
plot_flow(res, "infection", "Infection flow by age (infectiousness + susceptibility)", group=age)


## Partial stratification

Only `I` is severity-stratified (`stratify(severity, where=state["I"])`).
Incident infections fan out with an explicit `split=` via `update_flow`.
Infectiousness weights on severity would need `group_by=severity` on the FOI,
but susceptibles are not severity-stratified — the rate cannot align. Use a
whole-population FOI here; severity differences enter via death adjustments
and the incidence split.

In [ ]:
m = base.copy()
m.stratify(severity, where=state["I"])
m.update_flow(
    "infection",
    split={severity: {"asymptomatic": 0.3, "mild": 0.5, "severe": 0.2}},
)
m.adjust_flow(
    "infection_death",
    Multiply(0.5, where=severity["asymptomatic"]),
    Multiply(1.5, where=severity["severe"]),
)
pmap_sev = m.pmap
y0_sev = jnp.zeros(pmap_sev.size)
y0_sev = y0_sev.at[pmap_sev.select(state["S"])].set(990.0)
y0_sev = y0_sev.at[pmap_sev.select(state["I"] & severity["asymptomatic"])].set(10.0)
res = run(m, PropertyData.wrap(pmap_sev, y0_sev))
assert len(pmap_sev.select(state["I"])) == 3
plot_comp(res, "Severity stratification on I only")

In [ ]:
inc = res["infection"].sum_over(severity, side="dest").to_pandas().add_prefix("infection ")
death = (
    res["infection_death"].sum_over(severity, side="source").to_pandas().add_prefix("death ")
)
assert "infection severity=severe" in inc.columns
assert "death severity=severe" in death.columns
pd.concat([inc, death], axis=1).plot(
    title="Incidence split and death flows by severity",
    labels={"index": "time (days)", "value": "people / day"},
)


## Multiple stratifications

Age then severity. Infectiousness and susceptibility can depend on age;
severity split and death risk depend on severity.

In [ ]:
m = base.copy()
m.stratify(age)
m.stratify(severity, where=state["I"])
m.update_flow(
    "infection",
    rate=age_foi(
        infectiousness={age["young"]: 1.2, age["old"]: 1.0},
        normalize_infectiousness=None,
    ),
    split={severity: {"asymptomatic": 0.3, "mild": 0.5, "severe": 0.2}},
)
m.adjust_flow("infection", Multiply(2.0, where=age["young"]))
m.adjust_flow(
    "infection_death",
    Multiply(0.5, where=severity["asymptomatic"]),
    Multiply(1.5, where=severity["severe"]),
)
pmap_both = m.pmap
y0_both = jnp.zeros(pmap_both.size)
y0_both = y0_both.at[pmap_both.select(state["S"] & age["young"])].set(990.0 * 0.6)
y0_both = y0_both.at[pmap_both.select(state["S"] & age["old"])].set(990.0 * 0.4)
y0_both = y0_both.at[
    pmap_both.select(state["I"] & age["young"] & severity["asymptomatic"])
].set(10.0 * 0.6)
y0_both = y0_both.at[
    pmap_both.select(state["I"] & age["old"] & severity["asymptomatic"])
].set(10.0 * 0.4)
y0_both = PropertyData.wrap(pmap_both, y0_both)
res = run(m, y0_both)
assert pmap_both.size == 2 + 2 * 3 + 2  # S×2 + I×2×3 + R×2
plot_comp(res, "Age × severity stratification")

In [ ]:
inc = res["infection"].sum_over(severity, side="dest").to_pandas().add_prefix("infection ")
death = (
    res["infection_death"].sum_over(severity, side="source").to_pandas().add_prefix("death ")
)
assert "infection severity=severe" in inc.columns
assert "death severity=severe" in death.columns
pd.concat([inc, death], axis=1).plot(
    title="Infection (dest) and death (source) flows by severity",
    labels={"index": "time (days)", "value": "people / day"},
)


## Interdependent stratifications

Severity mix depends on age: younger infections skew asymptomatic; older skew
severe. Encode that with age-specific `Multiply` on the infection edges
(`Source(age)` × `Dest(severity)`).

In [ ]:
young_w = {"asymptomatic": 0.5, "mild": 0.4, "severe": 0.1}
old_w = {"asymptomatic": 0.1, "mild": 0.4, "severe": 0.5}
m = base.copy()
m.stratify(age)
m.stratify(severity, where=state["I"])
m.update_flow(
    "infection",
    rate=age_foi(),
    split={severity: {s: 1.0 / 3.0 for s in severity.traits}},
    adjust=[
        Multiply(young_w[s] / (1.0 / 3.0), where=Source(age["young"]) & Dest(severity[s]))
        for s in severity.traits
    ]
    + [
        Multiply(old_w[s] / (1.0 / 3.0), where=Source(age["old"]) & Dest(severity[s]))
        for s in severity.traits
    ],
)
m.adjust_flow(
    "infection_death",
    Multiply(0.5, where=severity["asymptomatic"]),
    Multiply(1.5, where=severity["severe"]),
)
res = run(m, y0_both)
severe_old = float(
    np.max(
        np.asarray(
            res["comp"].select(state["I"] & age["old"] & severity["severe"]).values.data
        )
    )
)
severe_young = float(
    np.max(
        np.asarray(
            res["comp"]
            .select(state["I"] & age["young"] & severity["severe"])
            .values.data
        )
    )
)
assert severe_old >= 0.0 and severe_young >= 0.0
plot_comp(res, "Age-dependent severity mix")

In [ ]:
inc = res["infection"].to_pandas()
assert inc.shape[1] == 6  # 2 ages × 3 dest severities
inc.plot(
    title="Infection flows (age → severity mix)",
    labels={"index": "time (days)", "value": "people / day"},
)


## Under `jax.jit`

Age-stratified SIR: differentiate through `Param("beta")` under a single
compiled run.

In [ ]:
def loss(beta):
    res = cm_age.run({"beta": beta}, y0_age, t0=0.0, t1=20.0, dt=0.1, save=plan_age, solver="euler")
    return jnp.sum(jnp.asarray(res["comp"].select(state["I"]).values.data))


jitted = jax.jit(loss)
val = jitted(jnp.asarray(2.0))
grad = jax.grad(loss)(jnp.asarray(2.0))
assert jnp.isfinite(val) and jnp.isfinite(grad)
np.testing.assert_allclose(val, loss(jnp.asarray(2.0)), rtol=1e-4)
print(f"loss={float(val):.4g}, grad={float(grad):.4g}")

## Summary

| Idea | summer2 | summer4 |
|---|---|---|
| Stratify | `Stratification` + `stratify_with` | `PropertyMap.stratify` (new map) or `FlowModel.stratify` (in place; `copy()` to branch) |
| Population split | `set_population_split` | initial `y0` shares |
| Flow adjustments | `set_flow_adjustments` | `adjust=[Multiply(...)]` / `Overwrite` / `adjust_flow` |
| Infectiousness | `add_infectiousness_adjustments` on strat | `ForceOfInfection(infectiousness=...)` |
| Partial strat | `compartments=[...]` | `stratify(..., where=...)` |